# Fase 1 — Adquisición y validación de datos
## Proyecto integrador "La hora dorada"

**Pregunta:** ¿cuánto tarda *en realidad* la población de una región en
llegar por carretera a un centro de salud con capacidad resolutiva
(categoría II-1 en adelante), y dónde están las peores brechas?

**Esta fase** construye un conjunto de datos geoespacial limpio, validado y
documentado de **demanda** (centros poblados, SIGMED/INEI) y **oferta**
(establecimientos RENIPRESS/SUSALUD), para los 3 departamentos declarados en
`config.md` — uno costero, uno andino y uno amazónico.

Estructura del notebook:

1. **Adquisición / carga** — ¿está el archivo en `data/raw/`? Si no, se copia
   de `_data/` (caché del curso) o se descarga; luego se lee a memoria con
   `pandas` / `geopandas` controlando el encoding (`utf-8` / `latin-1`).
2. **Limpieza y validación espacial** — estandarizar texto de categorías y
   estados; validar coordenadas (vacías/cero, intercambio lat/lon, signo de
   hemisferio, fuera de Perú); medir todo en `logs/quality_report.csv`.
3. **Filtrado de ámbito** — quedarse solo con oferta y demanda de los 3
   departamentos objetivo.
4. **Almacenamiento** — guardar los datos limpios en `data/processed/` como
   GeoPackage.
5. **Exploración con Folium** — un mapa interactivo por departamento.

Ningún parámetro (departamentos, rutas, categorías resolutivas, bbox) está
escrito a mano en una celda: todo sale de `config.md`.


## Paso 0 — Configuración

`config.md` es YAML (con extensión `.md` para leerlo como documentación).
Se carga tal cual; nada se copia a mano.


In [3]:
from __future__ import annotations

import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import yaml
from shapely.geometry import Point

pd.set_option("display.max_columns", 60)

REPO_ROOT = Path.cwd()                       # el notebook vive en la raíz del repo
with open(REPO_ROOT / "config.md", "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)


def ruta(rel: str) -> Path:
    p = Path(rel)
    return p if p.is_absolute() else REPO_ROOT / p


def departamentos_objetivo() -> dict[str, str]:
    d = CONFIG["departamentos"]
    return {"costero": d["costero"], "andino": d["andino"], "amazonico": d["amazonico"]}


def codigos_inei() -> dict[str, str]:
    codigos = CONFIG["departamentos"]["codigos_inei"]
    return {n: codigos[n] for n in departamentos_objetivo().values()}


for clave in ("raw", "processed", "outputs", "reports"):
    ruta(CONFIG["rutas"][clave]).mkdir(parents=True, exist_ok=True)

BBOX = CONFIG["validacion"]["bbox_peru"]
CAT_RESOLUTIVAS = set(CONFIG["categorias"]["resolutivas"])
CAT_NO_CONCLUYENTES = set(CONFIG["categorias"]["no_concluyentes"])
ESTADOS_ACTIVOS = {e.upper() for e in CONFIG["estado_operativo_valido"]}

print("Departamentos objetivo :", departamentos_objetivo())
print("Códigos INEI           :", codigos_inei())
print("Categorías resolutivas :", sorted(CAT_RESOLUTIVAS))
print("Estado operativo válido :", sorted(ESTADOS_ACTIVOS))
print("bbox Perú              :", BBOX)


Departamentos objetivo : {'costero': 'PIURA', 'andino': 'AYACUCHO', 'amazonico': 'UCAYALI'}
Códigos INEI           : {'PIURA': '20', 'AYACUCHO': '05', 'UCAYALI': '25'}
Categorías resolutivas : ['II-1', 'II-2', 'II-E', 'III-1', 'III-2', 'III-E']
Estado operativo válido : ['ACTIVO']
bbox Perú              : {'lon_min': -81.4, 'lon_max': -68.6, 'lat_min': -18.4, 'lat_max': -0.04}


## Paso 1 — Adquisición / carga

Para cada fuente, en este orden:

1. Si ya existe en `data/raw/`, se usa tal cual (idempotente).
2. Si no, se copia desde `_data/` — la copia provista para el curso, declarada
   como `cache_local` en `config.md`. Es justo el caso que pide el enunciado:
   *"si el portal no está disponible el día que lo ejecute, use la caché y
   documente la fecha"*.
3. Si tampoco hay caché, se intenta la descarga real (RENIPRESS: se busca el
   CSV mensual más reciente; OSM: descarga directa de Geofabrik). Si la fuente
   es un portal interactivo sin enlace directo (SIGMED, límites), se imprimen
   instrucciones en vez de fallar en silencio.

Cada intento queda en `data/raw/download_log.json`.


In [4]:
FORZAR_DESCARGA = False   # True -> vuelve a copiar/descargar aunque exista

LOG_PATH = ruta(CONFIG["rutas"]["log_descargas"])
UA = {"User-Agent": CONFIG["descargas"]["user_agent"]}
TIMEOUT = CONFIG["descargas"]["timeout_segundos"]
REINTENTOS = CONFIG["descargas"]["reintentos"]


def _ahora() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _cargar_log() -> dict:
    if LOG_PATH.exists():
        try:
            return json.loads(LOG_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {}


def _guardar_log(log: dict) -> None:
    LOG_PATH.write_text(json.dumps(log, indent=2, ensure_ascii=False), encoding="utf-8")


def _registrar(log, clave, **campos):
    log.setdefault(clave, {}).update({"ultima_verificacion": _ahora(), **campos})


def _copiar(origen: Path, destino: Path) -> int:
    destino.parent.mkdir(parents=True, exist_ok=True)
    if origen.suffix.lower() == ".shp":          # el shapefile arrastra .dbf/.shx/.prj/...
        total = 0
        for lado in origen.parent.glob(origen.stem + ".*"):
            destino_lado = destino.parent / (destino.stem + lado.suffix)
            shutil.copyfile(lado, destino_lado)
            total += destino_lado.stat().st_size
        return total
    shutil.copyfile(origen, destino)
    return destino.stat().st_size


def _descargar(url: str, destino: Path) -> int:
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_suffix(destino.suffix + ".part")
    total = 0
    with requests.get(url, headers=UA, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk); total += len(chunk)
    tmp.replace(destino)
    return total


def _reintentar(fn):
    ultimo = None
    for i in range(1, REINTENTOS + 1):
        try:
            return fn()
        except (requests.RequestException, OSError) as e:
            ultimo = e; print(f"    intento {i}/{REINTENTOS} falló: {e}")
    raise ultimo


def obtener(clave: str, cfg: dict, log: dict, force: bool):
    destino = ruta(cfg["archivo_local"])
    if destino.exists() and destino.stat().st_size > 0 and not force:
        print(f"[{clave}] ya está en {destino.relative_to(REPO_ROOT)}")
        _registrar(log, clave, estado="ya_existia", archivo_local=str(destino),
                   tamano_bytes=destino.stat().st_size)
        return

    cache = cfg.get("cache_local")
    if cache and ruta(cache).exists():
        n = _copiar(ruta(cache), destino)
        fecha = datetime.fromtimestamp(ruta(cache).stat().st_mtime, tz=timezone.utc).isoformat(timespec="seconds")
        print(f"[{clave}] copiado de caché {ruta(cache).name} -> {destino.relative_to(REPO_ROOT)} ({n/1e6:.1f} MB)")
        _registrar(log, clave, estado="copiado_desde_cache", archivo_local=str(destino),
                   cache_local=str(ruta(cache)), tamano_bytes=n, fecha_archivo_cache=fecha,
                   fecha_dato=cfg.get("fecha_cache"))
        return

    tipo = cfg.get("tipo_descarga")
    try:
        if clave == "renipress":
            print(f"[{clave}] buscando el CSV mensual más reciente en {cfg['url']} ...")
            html = _reintentar(lambda: requests.get(cfg["url"], headers=UA, timeout=TIMEOUT).text)
            cands = sorted(set(re.findall(cfg["patron_archivo"], html)))
            if not cands:
                raise RuntimeError("ningún enlace calza con patron_archivo")
            fecha_de = lambda s: datetime(*map(int, re.search(r"(\d{2})-(\d{2})-(\d{4})", s).group(3, 2, 1)))
            archivo = max(cands, key=fecha_de)
            url = "https://www.datosabiertos.gob.pe/sites/default/files/" + archivo
            n = _reintentar(lambda: _descargar(url, destino))
            print(f"[{clave}] descargado {archivo} ({n/1e6:.1f} MB)")
            _registrar(log, clave, estado="descargado", archivo_local=str(destino), url=url, tamano_bytes=n)
        elif tipo == "directo":
            n = _reintentar(lambda: _descargar(cfg["url"], destino))
            print(f"[{clave}] descargado ({n/1e6:.1f} MB)")
            _registrar(log, clave, estado="descargado", archivo_local=str(destino), url=cfg["url"], tamano_bytes=n)
        else:
            origen = cfg.get("url") or cfg.get("fuente")
            print(f"[{clave}] descarga MANUAL: {origen}\n    guardar en: {destino}")
            _registrar(log, clave, estado="manual_requerido", archivo_local=str(destino), url_origen=origen)
    except Exception as e:
        print(f"[{clave}] ERROR: {e}")
        _registrar(log, clave, estado="error", mensaje=str(e))


log_descargas = _cargar_log()
for clave, cfg in CONFIG["fuentes"].items():
    if clave == "limites_administrativos":
        for capa, cfg_capa in cfg["capas"].items():
            obtener(f"limites_{capa}", {**cfg_capa, "fuente": cfg.get("fuente"),
                                        "tipo_descarga": "manual"}, log_descargas, FORZAR_DESCARGA)
    else:
        obtener(clave, cfg, log_descargas, FORZAR_DESCARGA)
_guardar_log(log_descargas)

print(f"\nRegistro -> {LOG_PATH.relative_to(REPO_ROOT)}")
pd.DataFrame(log_descargas).T[["estado", "tamano_bytes"]]


[renipress] ya está en data\raw\renipress.csv
[sigmed] ya está en data\raw\centros_poblados.shp
[osm_peru] ya está en data\raw\peru-latest.osm.pbf
[limites_departamento] ya está en data\raw\limites_departamento.gpkg
[limites_provincia] ya está en data\raw\limites_provincia.gpkg
[limites_distrito] ya está en data\raw\limites_distrito.gpkg

Registro -> data\raw\download_log.json


,estado,tamano_bytes
renipress,ya_existia,19074859
sigmed,ya_existia,4295300
osm_peru,ya_existia,255851767
limites_administrativos.departamento,ya_existia,5492736
limites_administrativos.provincia,ya_existia,12111872
limites_administrativos.distrito,ya_existia,29106176
limites_departamento,ya_existia,5492736
limites_provincia,ya_existia,12111872
limites_distrito,ya_existia,29106176


### 1.1 — Lectura a memoria, controlando el encoding

`chardet` detecta el encoding real del archivo. Se intenta primero el
declarado en `config.md` (`utf-8-sig` para RENIPRESS); si falla, se cae al
que detecte `chardet` (típicamente `latin-1`) y **se anota en el informe**.


In [5]:
import chardet


def leer_csv_con_encoding(path: Path, sep: str, encoding_esperado: str) -> tuple[pd.DataFrame, dict]:
    """Devuelve (df, nota) — nota describe qué encoding se usó y por qué."""
    try:
        df = pd.read_csv(path, sep=sep, encoding=encoding_esperado, dtype=str, low_memory=False)
        return df, {"encoding_usado": encoding_esperado, "detalle": "decodifica con el encoding de config.md"}
    except UnicodeDecodeError:
        det = chardet.detect(path.read_bytes()[:200_000])
        enc = det["encoding"] or "latin-1"
        df = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, low_memory=False)
        return df, {"encoding_usado": enc,
                    "detalle": f"{path.name} no decodifica como {encoding_esperado}; "
                               f"chardet detectó {enc} (confianza {det['confidence']:.2f})"}


# --- Oferta: RENIPRESS (CSV) ---
cfg_r = CONFIG["fuentes"]["renipress"]
renipress_raw, nota_encoding_renipress = leer_csv_con_encoding(
    ruta(cfg_r["archivo_local"]), cfg_r["separador"], cfg_r["encoding"])
print(f"RENIPRESS      : {len(renipress_raw):>7} filas x {renipress_raw.shape[1]} columnas "
      f"({nota_encoding_renipress['encoding_usado']})")

# --- Demanda: centros poblados (shapefile, ya trae geometría de punto) ---
centros_raw = gpd.read_file(ruta(CONFIG["fuentes"]["sigmed"]["archivo_local"]))
centros_raw = centros_raw.set_crs(4326) if centros_raw.crs is None else centros_raw.to_crs(4326)
print(f"Centros poblados: {len(centros_raw):>7} filas x {centros_raw.shape[1]} columnas ({centros_raw.crs})")

# --- Límites distritales (para el chequeo punto-en-polígono y los mapas) ---
ruta_distritos = ruta(CONFIG["fuentes"]["limites_administrativos"]["capas"]["distrito"]["archivo_local"])
distritos_raw = gpd.read_file(ruta_distritos) if ruta_distritos.exists() else None
if distritos_raw is not None:
    distritos_raw = distritos_raw.set_crs(4326) if distritos_raw.crs is None else distritos_raw.to_crs(4326)
    print(f"Distritos      : {len(distritos_raw):>7} polígonos ({distritos_raw.crs})")
else:
    print("Distritos      : no disponibles (el chequeo punto-en-polígono quedará 'no evaluado')")

# --- Red vial OSM: solo se verifica que esté; se usa en la Fase 2 ---
ruta_pbf = ruta(CONFIG["fuentes"]["osm_peru"]["archivo_local"])
print(f"OSM Perú (.pbf): {'presente' if ruta_pbf.exists() else 'FALTA'} "
      f"({ruta_pbf.stat().st_size/1e6:.0f} MB)" if ruta_pbf.exists() else "")

renipress_raw.head(3)


RENIPRESS      :   35471 filas x 31 columnas (utf-8-sig)
Centros poblados:  153400 filas x 19 columnas (EPSG:4326)
Distritos      :    1890 polígonos (EPSG:4326)
OSM Perú (.pbf): presente (256 MB)


,INSTITUCION,COD_IPRESS,NOMBRE,CLASIFICACION,TIPO_ESTABLECIMIENTO,DEPARTAMENTO,PROVINCIA,DISTRITO,UBIGEO,DIRECCION,CO_DISA,COD_RED,COD_MICRORRED,DISA,RED,MICRORED,COD_UE,UNIDAD_EJECUTORA,CATEGORIA,TELEFONO,HORARIO,INICIO_ACTIVIDAD,ESTADO,NORTE,ESTE,IMAGEN_1,FE_ACT_IMAGEN_1,IMAGEN_2,FE_ACT_IMAGEN_2,IMAGEN_3,FE_ACT_IMAGEN_3
0,GOBIERNO REGIONAL,00002806,LA NOVIA,PUESTOS DE SALUD O POSTAS DE SALUD,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,CARRETERA IBERIA KM 80,25.0,137.0,915.0,MADRE DE DIOS,MADRE DE DIOS,IBERIA,879.0,SALUD MADRE DE DIOS,I-1,973267838,7:00 - 19:00,1995-01-01,ACTIVO,-11.8671856,-69.13774377,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN
1,GOBIERNO REGIONAL,00002807,SANTA MARIA,PUESTOS DE SALUD O POSTAS DE SALUD,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,CARRETERA IBERIA KM. 85,25.0,137.0,915.0,MADRE DE DIOS,MADRE DE DIOS,IBERIA,879.0,SALUD MADRE DE DIOS,I-1,931494291,7.00 -19:00,1995-05-01,ACTIVO,-11.89472628,-69.00601556,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN
2,GOBIERNO REGIONAL,00002808,P.S. CAMBRUNE,PUESTOS DE SALUD O POSTAS DE SALUD,ESTABLECIMIENTO DE SALUD SIN INTERNAMIENTO,MOQUEGUA,MARISCAL NIETO,CARUMAS,180102,CALLE 28 DE JULIO S/N,26.0,138.0,918.0,MOQUEGUA,MOQUEGUA,CARUMAS,884.0,SALUD MOQUEGUA,I-2,962749104,08:00-20:00,1963-10-21,ACTIVO,-16.82438155,-70.67862772,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN,http://app20.susalud.gob.pe:8080/registro-reni...,NaN


## Paso 2 — Limpieza y validación espacial

### 2.1 — Acumulador del Informe de Calidad de Datos

Cada regla llama a `qr.check(...)`. Nada se descarta en silencio: aunque la
acción sea *"no evaluado"*, queda una fila explicando por qué. Al final se
escribe `logs/quality_report.csv`.


In [6]:
class QualityReport:
    def __init__(self):
        self.filas: list[dict] = []

    def check(self, dataset, regla, evaluados, marcados, accion, motivo):
        pct = round(100 * marcados / evaluados, 2) if evaluados else None
        self.filas.append({"dataset": dataset, "regla": regla, "registros_evaluados": int(evaluados),
                           "registros_marcados": int(marcados), "porcentaje": pct,
                           "accion": accion, "motivo": motivo})
        print(f"  [{dataset}] {regla}: {marcados}/{evaluados}"
              + (f" ({pct}%)" if pct is not None else "") + f" -> {accion}")

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.filas)


qr = QualityReport()


### 2.2 — Estandarización de texto: categorías y estados

- **Estado**: `strip()` + `upper()` → se compara contra `estado_operativo_valido`
  de `config.md` (`ACTIVO`).
- **Categoría**: se ignoran mayúsculas y espacios; vacío / `"0"` / `"S/C"` →
  desconocida; una cadena tipo romano (`I`–`III`) + sufijo (`1/2/3/E`), con o
  sin guion, se reescribe a la forma canónica `"II-1"`. Cualquier otra cosa →
  desconocida (se conserva el dato crudo, pero no cuenta como oferta
  resolutiva). Las reglas están a la vista, en el código.
- **Mojibake** (`utf-8` vs `latin-1` en el contenido de las celdas): se
  revierte el patrón conocido de RENIPRESS (`ÿ`→`ñ`) y la doble codificación.


In [7]:
# --- categoría ---
_PAT_CAT = re.compile(r"^(I{1,3})-?(1|2|3|E)$")
_CAT_VACIA = {"", "0", "NAN", "NONE", "S/C", "SINCATEGORIA", "SINCATEGORÍA"}


def normalizar_categoria(raw) -> str | None:
    if raw is None:
        return None
    txt = re.sub(r"\s+", "", str(raw).strip().upper())
    if txt in _CAT_VACIA:
        return None
    m = _PAT_CAT.match(txt)
    return f"{m.group(1)}-{m.group(2)}" if m else None


def grupo_categoria(cat_norm) -> str:
    if cat_norm in CAT_RESOLUTIVAS:
        return "resolutiva"
    if cat_norm in CAT_NO_CONCLUYENTES:
        return "no_concluyente"
    return "desconocida"


def normalizar_estado(raw) -> str:
    return str(raw).strip().upper()


# --- mojibake en el contenido de columnas de texto ---
_MOJIBAKE = {"ÿ": "ñ", "Ÿ": "Ñ"}
_DOBLE_COD = ("Ã", "Â", "�")


def _arreglar_mojibake(v):
    if not isinstance(v, str) or not v:
        return v, False
    out, cambio = v, False
    for malo, bueno in _MOJIBAKE.items():
        if malo in out:
            out = out.replace(malo, bueno); cambio = True
    if any(x in out for x in _DOBLE_COD):
        try:
            out = out.encode("latin-1").decode("utf-8"); cambio = True
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
    return out, cambio


def arreglar_encoding_columnas(df, columnas, dataset):
    n = 0
    for c in columnas:
        if c not in df.columns:
            continue
        res = df[c].map(_arreglar_mojibake)
        df[c] = res.map(lambda r: r[0])
        n += int(sum(r[1] for r in res))
    qr.check(dataset, "encoding_texto_utf8_vs_latin1", len(df), n,
             "corregido" if n else "sin problemas", "mojibake conocido (ÿ→ñ) y doble codificación utf-8/latin-1")
    return df


### 2.3 — Validación de coordenadas

Sobre las columnas `NORTE` (latitud) y `ESTE` (longitud) de RENIPRESS —
nombres heredados de un esquema UTM, pero en la práctica traen **grados
decimales**. Reglas, en orden:

| # | Regla | Acción |
|---|---|---|
| 1 | latitud/longitud vacía o no numérica | eliminar (sin coordenada no hay ruta en Fase 2) |
| 2 | coordenada ≈ 0 (`|valor| < tol`) | eliminar (centinela de nulo, no una posición real) |
| 3 | **signo de hemisferio**: valor positivo cuyo negativo sí cae en Perú | corregir el signo (Perú está en lat/lon negativas) |
| 4 | **intercambio lat/lon**: `(lat,lon)` fuera de Perú pero `(lon,lat)` dentro | intercambiar los dos valores |
| 5 | sigue fuera del bbox de Perú | eliminar (no recuperable con una regla justificada) |

Para centros poblados, que ya traen geometría, aplican las equivalentes:
geometría vacía y fuera del bbox.


In [8]:
def validar_coordenadas_xy(df, lat_col, lon_col, dataset):
    """Limpia lat/lon numéricas in-place. Devuelve el df filtrado."""
    tol = CONFIG["validacion"]["tolerancia_coordenada_cero"]
    n0 = len(df)
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    lon = pd.to_numeric(df[lon_col], errors="coerce")

    # 1) faltantes / no numéricas
    falta = lat.isna() | lon.isna()
    qr.check(dataset, "coordenadas_vacias_o_no_numericas", n0, int(falta.sum()),
             "eliminado", "sin coordenada numérica no se puede rutear en la Fase 2")
    df, lat, lon = df.loc[~falta].copy(), lat.loc[~falta], lon.loc[~falta]

    # 2) cero (centinela de nulo)
    cero = lat.abs().lt(tol) | lon.abs().lt(tol)
    qr.check(dataset, "coordenadas_en_cero", n0, int(cero.sum()),
             "eliminado", f"|valor| < {tol} se trata como nulo, no como posición real cerca de (0,0)")
    df, lat, lon = df.loc[~cero].copy(), lat.loc[~cero], lon.loc[~cero]

    # 3) signo de hemisferio (Perú: lat y lon negativas)
    lat_pos_recuperable = lat.gt(0) & (-lat).between(BBOX["lat_min"], BBOX["lat_max"])
    lon_pos_recuperable = lon.gt(0) & (-lon).between(BBOX["lon_min"], BBOX["lon_max"])
    n_signo = int((lat_pos_recuperable | lon_pos_recuperable).sum())
    lat = lat.mask(lat_pos_recuperable, -lat)
    lon = lon.mask(lon_pos_recuperable, -lon)
    qr.check(dataset, "signo_de_hemisferio_invertido", n0, n_signo,
             "corregido", "coordenada positiva cuyo valor negativo sí cae dentro de Perú")

    # 4) intercambio lat/lon
    dentro = lat.between(BBOX["lat_min"], BBOX["lat_max"]) & lon.between(BBOX["lon_min"], BBOX["lon_max"])
    dentro_si_swap = lon.between(BBOX["lat_min"], BBOX["lat_max"]) & lat.between(BBOX["lon_min"], BBOX["lon_max"])
    swap = (~dentro) & dentro_si_swap
    qr.check(dataset, "coordenadas_lat_lon_intercambiadas", n0, int(swap.sum()),
             "corregido", "(lat,lon) cae fuera de Perú pero (lon,lat) cae dentro: se intercambian")
    lat_f = lat.mask(swap, lon)
    lon_f = lon.mask(swap, lat)

    # 5) sigue fuera del bbox -> eliminar
    fuera = ~(lat_f.between(BBOX["lat_min"], BBOX["lat_max"]) & lon_f.between(BBOX["lon_min"], BBOX["lon_max"]))
    qr.check(dataset, "coordenadas_fuera_de_peru", n0, int(fuera.sum()),
             "eliminado", f"fuera de lon[{BBOX['lon_min']},{BBOX['lon_max']}] x lat[{BBOX['lat_min']},{BBOX['lat_max']}]")
    df = df.loc[~fuera].copy()
    df["_lat"] = lat_f.loc[df.index]
    df["_lon"] = lon_f.loc[df.index]
    return df


def validar_geometria_puntos(gdf, dataset):
    n0 = len(gdf)
    mala = gdf.geometry.isna() | gdf.geometry.is_empty
    qr.check(dataset, "geometria_vacia", n0, int(mala.sum()), "eliminado", "sin geometría no se puede rutear")
    gdf = gdf.loc[~mala].copy()
    p = gdf.geometry.representative_point()
    dentro = p.y.between(BBOX["lat_min"], BBOX["lat_max"]) & p.x.between(BBOX["lon_min"], BBOX["lon_max"])
    qr.check(dataset, "coordenadas_fuera_de_peru", n0, int((~dentro).sum()),
             "eliminado", "el punto representativo cae fuera del bbox de Perú")
    return gdf.loc[dentro].copy()


def marcar_duplicados(df, id_col, dataset):
    n0 = len(df)
    dup = df[id_col].duplicated(keep="first")
    qr.check(dataset, "codigos_duplicados", n0, int(dup.sum()),
             "eliminado (se conserva la 1ª aparición)", f"'{id_col}' debe identificar un único registro")
    return df.loc[~dup].copy()


def _col(cols, candidatos):
    norm = {re.sub(r"[^A-Z0-9]", "", str(c).upper()): c for c in cols}
    for cand in candidatos:
        k = re.sub(r"[^A-Z0-9]", "", cand.upper())
        if k in norm:
            return norm[k]
    return None


### 2.4 — RENIPRESS: aplicar la limpieza

`resolutiva` = **estado activo** *y* **categoría en la lista blanca de
`config.md`** (`II-1, II-2, II-E, III-1, III-2, III-E`). Las categorías
`I-1`…`I-4` y las desconocidas se conservan en el dataset pero quedan
marcadas como no resolutivas (se excluyen del cálculo del más cercano en la
Fase 2).


In [9]:
DS_R = "renipress"
COLS_TEXTO_R = ["NOMBRE", "DIRECCION", "INSTITUCION", "UNIDAD_EJECUTORA",
                "DEPARTAMENTO", "PROVINCIA", "DISTRITO"]

qr.check(DS_R, "encoding_archivo", 1, 0 if nota_encoding_renipress["encoding_usado"] == cfg_r["encoding"] else 1,
         f"leído como {nota_encoding_renipress['encoding_usado']}", nota_encoding_renipress["detalle"])

r = renipress_raw.copy()
r = arreglar_encoding_columnas(r, COLS_TEXTO_R, DS_R)
r = validar_coordenadas_xy(r, "NORTE", "ESTE", DS_R)
r = marcar_duplicados(r, "COD_IPRESS", DS_R)

r["categoria_norm"] = r["CATEGORIA"].map(normalizar_categoria)
r["categoria_grupo"] = r["categoria_norm"].map(grupo_categoria)
qr.check(DS_R, "categoria_no_reconocida", len(r), int((r["categoria_grupo"] == "desconocida").sum()),
         "conservado con advertencia", "CATEGORIA vacía, '0' o que no calza con el patrón romano+sufijo")

r["estado_norm"] = r["ESTADO"].map(normalizar_estado)
r["operativo_activo"] = r["estado_norm"].isin(ESTADOS_ACTIVOS)
r["resolutiva"] = r["operativo_activo"] & (r["categoria_grupo"] == "resolutiva")

oferta = gpd.GeoDataFrame(r, geometry=gpd.points_from_xy(r["_lon"], r["_lat"]), crs="EPSG:4326")
print(f"\nRENIPRESS limpio: {len(oferta)} de {len(renipress_raw)} "
      f"({100*len(oferta)/len(renipress_raw):.1f}%) · resolutivos: {int(oferta['resolutiva'].sum())}")
oferta[["NOMBRE", "DEPARTAMENTO", "DISTRITO", "CATEGORIA", "categoria_norm",
        "categoria_grupo", "ESTADO", "resolutiva"]].head(8)


  [renipress] encoding_archivo: 0/1 (0.0%) -> leído como utf-8-sig
  [renipress] encoding_texto_utf8_vs_latin1: 4/35471 (0.01%) -> corregido
  [renipress] coordenadas_vacias_o_no_numericas: 13147/35471 (37.06%) -> eliminado
  [renipress] coordenadas_en_cero: 7/35471 (0.02%) -> eliminado
  [renipress] signo_de_hemisferio_invertido: 0/35471 (0.0%) -> corregido
  [renipress] coordenadas_lat_lon_intercambiadas: 0/35471 (0.0%) -> corregido
  [renipress] coordenadas_fuera_de_peru: 0/35471 (0.0%) -> eliminado
  [renipress] codigos_duplicados: 0/22317 (0.0%) -> eliminado (se conserva la 1ª aparición)
  [renipress] categoria_no_reconocida: 2988/22317 (13.39%) -> conservado con advertencia

RENIPRESS limpio: 22317 de 35471 (62.9%) · resolutivos: 599


,NOMBRE,DEPARTAMENTO,DISTRITO,CATEGORIA,categoria_norm,categoria_grupo,ESTADO,resolutiva
0,LA NOVIA,MADRE DE DIOS,TAHUAMANU,I-1,I-1,no_concluyente,ACTIVO,False
1,SANTA MARIA,MADRE DE DIOS,TAHUAMANU,I-1,I-1,no_concluyente,ACTIVO,False
2,P.S. CAMBRUNE,MOQUEGUA,CARUMAS,I-2,I-2,no_concluyente,ACTIVO,False
3,CENTRO DE SALUD CARUMAS,MOQUEGUA,CARUMAS,I-3,I-3,no_concluyente,ACTIVO,False
4,P.S. SACUAYA,MOQUEGUA,CUCHUMBAYA,I-2,I-2,no_concluyente,ACTIVO,False
5,SOCORRO,HUANCAVELICA,PAMPAS,I-2,I-2,no_concluyente,ACTIVO,False
6,DOS DE MAYO,HUANCAVELICA,ACRAQUIA,I-2,I-2,no_concluyente,ACTIVO,False
7,AHUAYCHA,HUANCAVELICA,AHUAYCHA,I-2,I-2,no_concluyente,ACTIVO,False


### 2.5 — Centros poblados: aplicar la limpieza


In [10]:
DS_C = "centros_poblados"
c = centros_raw.copy()
c = validar_geometria_puntos(c, DS_C)
c = arreglar_encoding_columnas(c, [col for col in c.columns if c[col].dtype == object and col != "geometry"], DS_C)

id_cp = _col(c.columns, ["CODCP", "CCPP", "COD_CCPP", "IDCCPP", "CODIGO"])
if id_cp:
    c = marcar_duplicados(c, id_cp, DS_C)
else:
    qr.check(DS_C, "codigos_duplicados", len(c), 0, "no evaluado", "no se identificó columna de código único")

demanda = c
print(f"Centros poblados limpio: {len(demanda)} de {len(centros_raw)} "
      f"({100*len(demanda)/len(centros_raw):.1f}%)")
demanda.head(5)


  [centros_poblados] geometria_vacia: 0/153400 (0.0%) -> eliminado
  [centros_poblados] coordenadas_fuera_de_peru: 0/153400 (0.0%) -> eliminado
  [centros_poblados] encoding_texto_utf8_vs_latin1: 0/153400 (0.0%) -> sin problemas
  [centros_poblados] codigos_duplicados: 0/153400 (0.0%) -> eliminado (se conserva la 1ª aparición)
Centros poblados limpio: 153400 de 153400 (100.0%)


,UBIGEO,DEP,PROV,DIST,CODCP,NOMCP,MNOMCP,CAPITAL,CON_IE,NIVEL,CPINEI,CPINEI2,FUENTE_INE,FUENTE_G,Z,XGD,YGD,Y_X_COORD,geometry
0,230303,TACNA,JORGE BASADRE,ITE,661851,Icuy,ICUY,0,0,NaN,2303030007,NaN,INEI17,INEI,22,-71.113285,-17.837672,-17.8376716669999-71.113285,POINT (-71.11328 -17.83767)
1,230303,TACNA,JORGE BASADRE,ITE,677680,Punta Picata,PUNTA PICATA,0,0,NaN,2303030021,NaN,INEI17,INEI,11,-71.095762,-17.866908,-17.866908333-71.095761667,POINT (-71.09576 -17.86691)
2,230303,TACNA,JORGE BASADRE,ITE,240081,Tacahuay,TACAHUAY,0,0,NaN,2303030008,NaN,NaN,IGN,340,-71.099018,-17.799934,-17.7999338-71.09901827,POINT (-71.09902 -17.79993)
3,230303,TACNA,JORGE BASADRE,ITE,615979,Esquilimache,ESQUILIMACHE,0,0,NaN,NaN,NaN,NaN,INEI,161,-71.091930,-17.833363,-17.83336329-71.09192963,POINT (-71.09193 -17.83336)
4,180301,MOQUEGUA,ILO,ILO,214980,Icuy,ICUY,0,0,NaN,NaN,NaN,NaN,IGN,141,-71.128033,-17.811909,-17.81190929-71.1280327699999,POINT (-71.12803 -17.81191)


### 2.6 — Chequeo espacial: ¿cada punto cae dentro de su distrito?

*"Los puntos fuera del polígono de su distrito quedan registrados por ellos
mismos"* — no se eliminan (el polígono puede ser el impreciso), quedan
marcados y volcados a un CSV de detalle en `logs/`.


In [11]:
def marcar_fuera_de_su_distrito(gdf_pts, gdf_dist, ubigeo_pts, ubigeo_dist, dataset, ruta_detalle):
    gdf_pts = gdf_pts.reset_index(drop=True)
    dist = gdf_dist[[ubigeo_dist, "geometry"]].rename(columns={ubigeo_dist: "_ubg_poly"})
    unido = gpd.sjoin(gdf_pts, dist, how="left", predicate="within")
    unido = unido[~unido.index.duplicated(keep="first")].reindex(gdf_pts.index)
    fuera = unido["_ubg_poly"].isna() | (unido["_ubg_poly"].astype(str) != gdf_pts[ubigeo_pts].astype(str))
    gdf_pts = gdf_pts.copy()
    gdf_pts["fuera_de_su_distrito"] = fuera.to_numpy()
    qr.check(dataset, "punto_fuera_de_su_poligono_distrital", len(gdf_pts), int(fuera.sum()),
             "conservado con advertencia", "el punto no cae en el polígono del distrito que declara su UBIGEO")
    if fuera.any():
        cols = [x for x in [ubigeo_pts, "NOMBRE", "NOMCP", "DEPARTAMENTO", "DEP", "DISTRITO", "DIST"] if x in gdf_pts.columns]
        gdf_pts.loc[fuera, cols].to_csv(ruta_detalle, index=False, encoding="utf-8-sig")
        print(f"    detalle -> {ruta_detalle.relative_to(REPO_ROOT)}")
    return gdf_pts


LOGS = ruta(CONFIG["rutas"]["reports"])
ubigeo_dist = _col(distritos_raw.columns, ["UBIGEO", "IDDIST", "COD_DIST"]) if distritos_raw is not None else None

if distritos_raw is not None and ubigeo_dist:
    oferta = marcar_fuera_de_su_distrito(oferta, distritos_raw, "UBIGEO", ubigeo_dist, DS_R,
                                         LOGS / "renipress_fuera_de_poligono.csv")
    ubigeo_cp = _col(demanda.columns, ["UBIGEO", "COD_UBIGEO"])
    if ubigeo_cp:
        demanda = marcar_fuera_de_su_distrito(demanda, distritos_raw, ubigeo_cp, ubigeo_dist, DS_C,
                                              LOGS / "centros_poblados_fuera_de_poligono.csv")
else:
    qr.check(DS_R, "punto_fuera_de_su_poligono_distrital", len(oferta), 0, "no evaluado",
             "no hay capa de límites distritales en data/raw/")

oferta[oferta.get("fuera_de_su_distrito", False) == True][["NOMBRE", "DEPARTAMENTO", "DISTRITO"]].head(8)


  [renipress] punto_fuera_de_su_poligono_distrital: 1683/22317 (7.54%) -> conservado con advertencia
    detalle -> logs\renipress_fuera_de_poligono.csv
  [centros_poblados] punto_fuera_de_su_poligono_distrital: 811/153400 (0.53%) -> conservado con advertencia
    detalle -> logs\centros_poblados_fuera_de_poligono.csv


,NOMBRE,DEPARTAMENTO,DISTRITO
32,FONAVI IV,ICA,SUBTANJALLA
47,P.S. CHACLAYA,MOQUEGUA,UBINAS
52,CONSULTORIOS MEDICOS LIDER MEDIC EIRL,LIMA,SAN MARTIN DE PORRES
56,PUESTO DE SALUD CCARHUACCPAMPA,AYACUCHO,PARAS
65,P.S. QUINSACHATA,MOQUEGUA,UBINAS
70,PUESTO DE SALUD CONCHACHIRI,TACNA,TARATA
78,CCASAPATA,HUANCAVELICA,YAULI
87,COLLACACHI,PUNO,PICHACANI


## Paso 3 — Filtrado de ámbito

Quedarse solo con la oferta y la demanda de los 3 departamentos de
`config.md`. RENIPRESS se filtra por nombre de `DEPARTAMENTO`; los centros
poblados, por el prefijo de 2 dígitos del `UBIGEO` (código INEI del
departamento).


In [12]:
DEPTOS = departamentos_objetivo()          # {'costero': 'PIURA', ...}
CODIGOS = codigos_inei()                    # {'PIURA': '20', ...}
NOMBRES = set(DEPTOS.values())

oferta_ambito = oferta[oferta["DEPARTAMENTO"].str.upper().isin(NOMBRES)].copy()

ubigeo_cp = _col(demanda.columns, ["UBIGEO", "COD_UBIGEO"])
prefijos = tuple(CODIGOS.values())
demanda_ambito = demanda[demanda[ubigeo_cp].astype(str).str.zfill(6).str.startswith(prefijos)].copy() \
    if ubigeo_cp else demanda.iloc[0:0]

resumen = []
for rol, dep in DEPTOS.items():
    o = oferta_ambito[oferta_ambito["DEPARTAMENTO"].str.upper() == dep]
    d = demanda_ambito[demanda_ambito[ubigeo_cp].astype(str).str.zfill(6).str.startswith(CODIGOS[dep])]
    resumen.append({"rol": rol, "departamento": dep, "establecimientos": len(o),
                    "resolutivos": int(o["resolutiva"].sum()), "centros_poblados": len(d)})

print(f"Oferta en ámbito : {len(oferta_ambito)}   Demanda en ámbito: {len(demanda_ambito)}")
pd.DataFrame(resumen)


Oferta en ámbito : 2301   Demanda en ámbito: 17187


,rol,departamento,establecimientos,resolutivos,centros_poblados
0,costero,PIURA,1264,29,4325
1,andino,AYACUCHO,708,11,11144
2,amazonico,UCAYALI,329,6,1718


## Paso 4 — Almacenamiento

GeoPackage en `data/processed/`: una capa nacional limpia de cada tabla y un
recorte por cada uno de los 3 departamentos de `config.md`. Cambiar esos 3
nombres en `config.md` cambia automáticamente qué se recorta y se guarda.
Además se escribe el **Informe de Calidad de Datos** (`logs/quality_report.csv`).


In [13]:
PROC = ruta(CONFIG["rutas"]["processed"])

oferta.to_file(PROC / "renipress_nacional.gpkg", driver="GPKG")
demanda.to_file(PROC / "centros_poblados_nacional.gpkg", driver="GPKG")
print(f"nacional -> renipress ({len(oferta)}) · centros_poblados ({len(demanda)})")

for dep in DEPTOS.values():
    o = oferta_ambito[oferta_ambito["DEPARTAMENTO"].str.upper() == dep]
    d = demanda_ambito[demanda_ambito[ubigeo_cp].astype(str).str.zfill(6).str.startswith(CODIGOS[dep])]
    o.to_file(PROC / f"renipress_{dep.lower()}.gpkg", driver="GPKG")
    if len(d):
        d.to_file(PROC / f"centros_poblados_{dep.lower()}.gpkg", driver="GPKG")
    print(f"  {dep:<9}: renipress {len(o):>4}  ·  centros_poblados {len(d):>6}")

# --- Informe de Calidad de Datos ---
reporte = qr.to_frame()
ruta_csv = LOGS / "quality_report.csv"
reporte.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

lineas = ["# Informe de Calidad de Datos — Fase 1", f"Generado: {_ahora()}", "",
          "| Dataset | Regla | Evaluados | Marcados | % | Acción | Motivo |",
          "|---|---|---:|---:|---:|---|---|"]
for _, x in reporte.iterrows():
    pct = "—" if pd.isna(x["porcentaje"]) else f"{x['porcentaje']}%"
    lineas.append(f"| {x['dataset']} | {x['regla']} | {x['registros_evaluados']} | "
                  f"{x['registros_marcados']} | {pct} | {x['accion']} | {x['motivo'].replace('|', '/')} |")
(LOGS / "quality_report.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")

print(f"\nInforme -> {ruta_csv.relative_to(REPO_ROOT)}  (+ quality_report.md)")
reporte


nacional -> renipress (22317) · centros_poblados (153400)
  PIURA    : renipress 1264  ·  centros_poblados   4325
  AYACUCHO : renipress  708  ·  centros_poblados  11144
  UCAYALI  : renipress  329  ·  centros_poblados   1718

Informe -> logs\quality_report.csv  (+ quality_report.md)


,dataset,regla,registros_evaluados,registros_marcados,porcentaje,accion,motivo
0,renipress,encoding_archivo,1,0,0.00,leído como utf-8-sig,decodifica con el encoding de config.md
1,renipress,encoding_texto_utf8_vs_latin1,35471,4,0.01,corregido,mojibake conocido (ÿ→ñ) y doble codificación u...
2,renipress,coordenadas_vacias_o_no_numericas,35471,13147,37.06,eliminado,sin coordenada numérica no se puede rutear en ...
3,renipress,coordenadas_en_cero,35471,7,0.02,eliminado,"|valor| < 1e-06 se trata como nulo, no como po..."
4,renipress,signo_de_hemisferio_invertido,35471,0,0.00,corregido,coordenada positiva cuyo valor negativo sí cae...
5,renipress,coordenadas_lat_lon_intercambiadas,35471,0,0.00,corregido,"(lat,lon) cae fuera de Perú pero (lon,lat) cae..."
6,renipress,coordenadas_fuera_de_peru,35471,0,0.00,eliminado,"fuera de lon[-81.4,-68.6] x lat[-18.4,-0.04]"
7,renipress,codigos_duplicados,22317,0,0.00,eliminado (se conserva la 1ª aparición),'COD_IPRESS' debe identificar un único registro
8,renipress,categoria_no_reconocida,22317,2988,13.39,conservado con advertencia,"CATEGORIA vacía, '0' o que no calza con el pat..."
9,centros_poblados,geometria_vacia,153400,0,0.00,eliminado,sin geometría no se puede rutear


## Paso 5 — Exploración con Folium

Un mapa interactivo por departamento, al estilo de
`referencias/Folium_Mapas_Interactivos_Peru.ipynb`: fondo claro, `MarkerCluster`
para agrupar muchos puntos, ficha HTML (`branca.IFrame`) en cada
establecimiento resolutivo y `LayerControl` para alternar capas.

| Capa | Contenido | Visible al abrir |
|---|---|---|
| Límites distritales | polígonos INEI del departamento | sí |
| Centros poblados (demanda) | clúster de centros poblados | sí |
| No resolutivos I-1 a I-4 | establecimientos sin capacidad resolutiva | no |
| Resolutivos II-1+ | establecimientos con capacidad resolutiva | sí |

Cada mapa se guarda como HTML independiente en `data/outputs/`.


In [14]:
import folium as fm
from folium.plugins import MarkerCluster, FastMarkerCluster
import branca

COLOR_RES   = "#d00000"   # rojo  — II-1+
COLOR_NORES = "#8d99ae"   # gris  — I-1 a I-4 / desconocida
COLOR_DEM   = "#2a9d8f"   # verde — centros poblados
COLOR_LIM   = "#1d3557"   # azul  — borde de distritos
# Esri World Light Gray Canvas: fondo claro y minimalista, gratuito y sin API key
# (los atajos "cartodbpositron" de Folium ahora exigen key de CARTO).
TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
TILES_ATTR = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"


def _v(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "—"
    return str(x).replace("`", "'")          # el backtick rompe los tooltips de Folium


def _popup(row):
    et, vl = "#1d3557", "#eef2f7"
    campos = [("Categoría", row.get("categoria_norm")), ("Estado", row.get("ESTADO")),
              ("Institución", row.get("INSTITUCION")), ("Provincia", row.get("PROVINCIA")),
              ("Distrito", row.get("DISTRITO")), ("Dirección", row.get("DIRECCION"))]
    trs = "".join(f'<tr><td style="background:{et};color:#fff;padding:5px 8px;font-size:11px">{k}</td>'
                  f'<td style="background:{vl};padding:5px 8px;font-size:11px">{_v(v)}</td></tr>' for k, v in campos)
    html = ('<table style="width:320px;border-collapse:collapse;font-family:Arial,sans-serif">'
            f'<tr><td colspan="2" style="background:#03071e;color:#fff;font-weight:bold;font-size:12px;'
            f'padding:8px;text-align:center">&#127973; {_v(row.get("NOMBRE"))}</td></tr>{trs}</table>')
    return fm.Popup(branca.element.IFrame(html=html, width=350, height=230), parse_html=True)


def _banner(dep):
    return ('<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:9999;'
            'background:#03071e;color:#fff;padding:6px 18px;border-radius:6px;font-weight:bold;'
            f'font-family:Arial,sans-serif;font-size:14px">La hora dorada &middot; {dep.title()}</div>')


def _leyenda(dep, nres, nnores, ncp):
    f = '<div><span style="color:{c};font-size:15px">&#9679;</span>&nbsp;{t}</div>'
    return ('<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:#fff;padding:11px 13px;'
            'border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);font-family:Arial,sans-serif;font-size:12px;'
            f'line-height:1.7"><div style="font-weight:bold;margin-bottom:3px">{dep.title()}</div>'
            + f.format(c=COLOR_RES, t=f"Resolutivo II-1+ ({nres})")
            + f.format(c=COLOR_NORES, t=f"No resolutivo I-1 a I-4 ({nnores})")
            + f.format(c=COLOR_DEM, t=f"Centros poblados ({ncp:,})")
            + '<div style="margin-top:4px;color:#666">RENIPRESS/SUSALUD &middot; SIGMED/INEI</div></div>')


def mapa_departamento(dep, codigo):
    o = oferta_ambito[oferta_ambito["DEPARTAMENTO"].str.upper() == dep]
    res, nores = o[o["resolutiva"]], o[~o["resolutiva"]]
    d = demanda_ambito[demanda_ambito[ubigeo_cp].astype(str).str.zfill(6).str.startswith(codigo)]
    lim = distritos_raw[distritos_raw["ccdd"].astype(str).str.zfill(2) == str(codigo)] \
        if distritos_raw is not None and "ccdd" in distritos_raw.columns else None

    marco = lim if (lim is not None and len(lim)) else o
    minx, miny, maxx, maxy = marco.total_bounds
    ancho = max(maxx - minx, maxy - miny)
    zoom = 6 if ancho > 4 else 7 if ancho > 2 else 8
    # centro + zoom fijos (nada de fit_bounds: al ejecutarse antes de que el
    # contenedor tenga tamaño, Leaflet calcula un zoom 0 y el mapa sale del
    # tamaño del mundo).
    m = fm.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2],
               tiles=TILES, attr=TILES_ATTR, zoom_start=zoom, control_scale=True)

    if lim is not None and len(lim):
        lim_dib = lim[["nombdist", "nombprov", "geometry"]].copy()
        lim_dib["geometry"] = lim_dib.geometry.simplify(0.003, preserve_topology=True)
        fm.GeoJson(lim_dib.to_json(), name="Límites distritales",
                   style_function=lambda _f: {"color": COLOR_LIM, "weight": 1, "fillOpacity": 0.03},
                   highlight_function=lambda _f: {"weight": 2.5, "fillOpacity": 0.12},
                   tooltip=fm.GeoJsonTooltip(fields=["nombdist", "nombprov"], aliases=["Distrito", "Provincia"],
                                             style="background:#fff;color:#333;font-family:Arial;font-size:12px;padding:8px")
                   ).add_to(m)

    if len(d):
        coords = [[round(g.y, 5), round(g.x, 5)] for g in d.geometry]
        fg = fm.FeatureGroup(name=f"Centros poblados — demanda ({len(d):,})", show=True)
        FastMarkerCluster(coords).add_to(fg)
        fg.add_to(m)

    fg_nr = fm.FeatureGroup(name=f"No resolutivos I-1 a I-4 ({len(nores)})", show=False)
    mc = MarkerCluster().add_to(fg_nr)
    for g, nom, cat in zip(nores.geometry, nores["NOMBRE"], nores["categoria_norm"]):
        fm.CircleMarker([g.y, g.x], radius=3, weight=0, fill=True, fill_color=COLOR_NORES,
                        fill_opacity=0.6, tooltip=f"{_v(nom)} · {_v(cat) if cat else 's/categoría'}").add_to(mc)
    fg_nr.add_to(m)

    fg_r = fm.FeatureGroup(name=f"Resolutivos II-1+ ({len(res)})", show=True)
    for _, row in res.iterrows():
        fm.CircleMarker([row.geometry.y, row.geometry.x], radius=8, color="#fff", weight=2, fill=True,
                        fill_color=COLOR_RES, fill_opacity=0.95,
                        tooltip=f"★ {_v(row['NOMBRE'])} ({_v(row['categoria_norm'])})",
                        popup=_popup(row)).add_to(fg_r)
    fg_r.add_to(m)

    fm.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(fm.Element(_banner(dep)))
    m.get_root().html.add_child(fm.Element(_leyenda(dep, len(res), len(nores), len(d))))
    return m


OUT_DIR = ruta(CONFIG["rutas"]["outputs"])
mapas = {}
for rol, dep in DEPTOS.items():
    m = mapa_departamento(dep, CODIGOS[dep])
    destino = OUT_DIR / f"mapa_acceso_{dep.lower()}.html"
    m.save(str(destino))
    mapas[dep] = m
    print(f"  {rol:>9} · {dep:<9} -> {destino.relative_to(REPO_ROOT)}")


    costero · PIURA     -> data\outputs\mapa_acceso_piura.html
     andino · AYACUCHO  -> data\outputs\mapa_acceso_ayacucho.html
  amazonico · UCAYALI   -> data\outputs\mapa_acceso_ucayali.html


In [17]:
from IPython.display import display

for dep, m in mapas.items():
    print(f"\n=== {dep} ===")
    display(m)



=== PIURA ===



=== AYACUCHO ===



=== UCAYALI ===


## Próximo paso — Fase 2 (enrutamiento)

Con `data/processed/renipress_<depto>.gpkg` y
`data/processed/centros_poblados_<depto>.gpkg` para los 3 departamentos, la
Fase 2 construye el grafo vial desde `data/raw/peru-latest.osm.pbf` y calcula
el tiempo real por carretera desde cada centro poblado hasta el
establecimiento **resolutivo** más cercano.
